# AutoMotion Mobile V10 — Renderizador

Este notebook recebe o ZIP criado pelo editor HTML, encontra automaticamente o vídeo e o projeto JSON e faz a renderização no Google Colab.

In [ ]:
from google.colab import files
import os, zipfile, json, shutil, glob, subprocess

print('Envie o AutoMotion_Projeto.zip')
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
if not zip_name:
    raise ValueError('Você precisa enviar um arquivo .zip criado pelo AutoMotion.')

ROOT = '/content/automotion_project'
shutil.rmtree(ROOT, ignore_errors=True)
os.makedirs(ROOT, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(ROOT)

print('ZIP descompactado com sucesso.')
print('Arquivos:', os.listdir(ROOT))


In [ ]:
video_ext = {'.mp4','.mov','.webm','.mkv','.m4v','.avi'}
videos = [p for p in glob.glob(ROOT + '/*') if os.path.splitext(p)[1].lower() in video_ext]
project_path = os.path.join(ROOT, 'projeto.json')
if not videos:
    raise FileNotFoundError('Nenhum vídeo foi encontrado dentro do ZIP.')
if not os.path.exists(project_path):
    raise FileNotFoundError('projeto.json não foi encontrado dentro do ZIP.')
video_path = videos[0]
with open(project_path, 'r', encoding='utf-8') as f:
    project = json.load(f)
print('Vídeo:', video_path)
print('Visualizações:', len(project.get('edits', [])))
print(json.dumps(project, ensure_ascii=False, indent=2))


In [ ]:
subprocess.run(['ffmpeg','-version'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
print('FFmpeg está disponível no Colab.')


In [ ]:
output_path = '/content/AutoMotion_final.mp4'
cmd = ['ffmpeg','-y','-i',video_path,'-c:v','libx264','-preset','fast','-crf','18','-c:a','aac','-b:a','192k',output_path]
subprocess.run(cmd, check=True)
print('Renderização concluída:', output_path)


In [ ]:
from google.colab import files
files.download('/content/AutoMotion_final.mp4')
